In [ ]:
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import matplotlib as mpl
import matplotlib.patheffects as mpe
import matplotlib.pyplot as plt
import numpy as np
import optuna
from pathlib import Path
import pandas as pd
from scipy import stats
import os
import rasterio as rio
from glob import glob
from scipy.stats import mannwhitneyu
import seaborn as sns
import torch
from torch_geometric.utils import k_hop_subgraph
from torch_sparse import SparseTensor
from tqdm import tqdm
import xarray as xr

from project.utils.data import enrich, load_hillshade, sel_subset
from project.utils.metrics import compute_metrics
from project.utils.plotting import ProjectCustomExtent, despine, cbar_halo

ZH = True

plt.rcParams["figure.dpi"] = 150


# Switzerland LV95 with extended limits to cover 'Hydrological Switzerland'.
if ZH:
    lv95 = ProjectCustomExtent(2056, [2665610, 2757188, 1185302, 1276486])
else:
    lv95 = ProjectCustomExtent(2056, [2474273, 2842000, 1058000, 1325000])

In [ ]:
# Load predictions.
ds = xr.open_zarr('/net/argon/landclim/kraftb/machflow_routing/data/results/predictions.zarr').sel(
    time=slice('1993', None), model='routed_random')[['Q_pred_m3s', 'prec', 'is_lake', 'area']].drop_vars(
        ['x', 'y', 'model', 'hru_i'])

# Load HRU polygons.
hrus = gpd.read_file(
    '/net/argon/landclim/kraftb/machflow_routing/data/processed/hrus/hrus.gpkg', use_arrow=True).set_index('EZGNR')

hrus_lakes = hrus.loc[ds.hru.where(ds.is_lake.load(), drop=True)]

rivers = gpd.read_file(
    '/net/argon/landclim/kraftb/machflow_routing/data/raw/rivernet', use_arrow=True)[['CUM_LEN', 'geometry']]

bg = load_hillshade(
    hillshade_dir='/net/argon/landclim/kraftb/machflow_routing/data/processed/',
    hrus=hrus,
    ds=ds,
    use_terrain=False,
    target_crs='EPSG:2056',
    lake_rgb=(86, 150, 245),
    force_recompute=False)

# rivers_cut = gpd.overlay(
#     rivers,
#     hrus.reset_index()[['EZGNR', 'geometry']],
#     how='intersection'
# ).set_index('EZGNR', drop=False)

# common_hrus_poly = hrus.index.intersection(ds.hru.values)
# common_hrus_rivers = rivers_cut.index.intersection(ds.hru.values)

# rivers_cut = rivers_cut.loc[common_hrus_rivers]

# rivers_cut = rivers_cut.explode(index_parts=False).reset_index(drop=True)


In [ ]:
if ZH:
    limmat = 149567
    toess = 161368
    glatt = 178289

    H1 = hrus.loc[toess].H1
    H2 = hrus.loc[toess].H2
    toess_hru_mask = (hrus.H1 >= H1) & (hrus.H2 < H2)

    H1 = hrus.loc[limmat].H1
    H2 = hrus.loc[limmat].H2
    limmat_hru_mask = (hrus.H1 >= H1) & (hrus.H2 < H2)

    H1 = hrus.loc[glatt].H1
    H2 = hrus.loc[glatt].H2
    glatt_hru_mask = (hrus.H1 >= H1) & (hrus.H2 < H2)

    hru_mask = toess_hru_mask | limmat_hru_mask | glatt_hru_mask

    hrus = hrus.loc[hru_mask]
    ds = ds.sel(hru=hrus.index)

In [ ]:
def parametric_return_period(data, dist_name='gumbel_r', fit=True, params=None, max_rp_years=1000):
    """
    Compute return periods in the same order as the input array.
    """
    data = np.asarray(data)
    mask = np.isfinite(data)
    clean_data = data[mask]

    if len(clean_data) == 0:
        return np.full_like(data, np.nan, dtype=float)

    dist = getattr(stats, dist_name)

    if fit:
        params = dist.fit(clean_data)
    elif params is None:
        raise ValueError("You must either set fit=True or provide 'params'.")

    fitted_dist = dist(*params)
    cdf = fitted_dist.cdf(clean_data)

    # Clip to avoid inf
    eps = 1e-12
    cdf = np.clip(cdf, 0, 1 - eps)
    rp_clean = 1.0 / (1.0 - cdf) / 365.25

    # Cap unrealistically large return periods
    rp_clean = np.clip(rp_clean, None, max_rp_years)

    # Put return periods back into original array order
    rp = np.full_like(data, np.nan, dtype=float)
    rp[mask] = rp_clean

    return rp

In [ ]:
Q = ds.Q_pred_m3s.load()
Q_ano = Q - Q.median('time').compute()

In [ ]:
from scipy.stats import rankdata

def pct_rank(a):
    # Return percentile rank (0–1)
    return (rankdata(a, method='average') - 1) / (len(a) - 1)

ranks = xr.apply_ufunc(
    pct_rank,
    Q_ano,
    input_core_dims=[['time']],
    output_core_dims=[['time']],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[ds.Q_pred_m3s.dtype],
).compute()

pct_pm = (ranks - 0.5) * 200

In [ ]:
ds['Q_ano'] = pct_pm 

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import matplotlib as mpl
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
import os

def plot_time_step(i, time, ds, hrus, rivers, bg, lv95, plot_prec: bool = False):
    dss = ds.sel(time=time)
    hrus_p = enrich(dss.prec, hrus, 'p')
    
    fig, ax = plt.subplots(figsize=(4, 4) if ZH else (7, 4), subplot_kw={'projection': lv95, 'frameon': True})
    ax.set_global()

    if plot_prec:
        # Plot precipitation
        cmap, norm, alpha = 'Greys', mpl.colors.Normalize(vmin=0.0, vmax=50), 0.4
        hrus_p.plot(column='p', ec='w', lw=0, norm=norm, cmap=cmap, alpha=alpha,
                    antialiased=True, legend=False, ax=ax, rasterized=True)

        if ZH:
            axins = ax.inset_axes((0.76, 0.93, 0.18, 0.02), zorder=25)
        else:
            axins = ax.inset_axes((0.03, 0.93, 0.18, 0.02), zorder=25)
        mappable = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
        cbar = plt.colorbar(mappable, cax=axins, orientation='horizontal', extend='max', label='Precipitation (mm/d)')
        cbar.ax.xaxis.label.set_size(7)
        cbar.ax.tick_params(labelsize=7)
        cbar_halo(cbar=cbar, linewidth=0, labelsize=6)

    # Background
    bg.plot.imshow(ax=ax, interpolation='bilinear', alpha=0.5, zorder=0)
    
    # Rivers
    var = 'Q_ano'
    hrus_T = enrich(dss, hrus, var)
    rivers_cut = gpd.overlay(rivers, hrus_T.reset_index()[['EZGNR', var, 'geometry']], how='intersection')
    rivers_Q = rivers_cut.explode(index_parts=False).reset_index(drop=True)
    rivers_Q = rivers_Q.sort_values(var, ascending=True)
    region_adjust = +0.3 if ZH else -0.1
    rw = np.clip(np.log(rivers_Q.CUM_LEN + 1) / 15 + region_adjust, 0.3, None)
    
    cmap, norm = 'coolwarm', mpl.colors.Normalize(vmin=-100, vmax=100)
    rivers_Q.plot(color='0.3', linewidth=rw + 0.4 if ZH else rw + 0.2, zorder=2, capstyle='round', legend=False, ax=ax)
    rivers_Q.plot(column=var, linewidth=rw - 0.3, norm=norm, zorder=2, capstyle='round', fc='none', alpha=1.0, cmap=cmap, legend=False, ax=ax)

    if ZH:
        axins = ax.inset_axes((0.70, 0.80, 0.18, 0.02), zorder=25)
    else:
        axins = ax.inset_axes((0.03, 0.80, 0.18, 0.02), zorder=25)
    mappable = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    cbar = plt.colorbar(mappable, cax=axins, orientation='horizontal', extend='neither', label='Relative discharge')
    cbar.set_ticks([-100, 0, 100])
    cbar.ax.set_xticklabels(['very\nlow', 'normal', 'very\nhigh'])
    cbar_halo(cbar=cbar, linewidth=0, labelsize=6)
    
    date = dss.time.dt.strftime('%Y-%m-%d').item()
    ax.text(0.99, 0.01, f'{date}', fontsize=8, transform=ax.transAxes,
            verticalalignment='bottom', horizontalalignment='right', zorder=40)
    
    ax.set_title('')
    a = '_zh' if ZH else '_ch'
    fig.savefig(f'./figures/anim/{i:04d}{a}.png', dpi=250, bbox_inches='tight')
    plt.close()


In [ ]:
# Wrapper function
def plot_wrapper(args):
    i, time, ds, hrus, rivers, bg, lv95 = args
    plot_time_step(i, time, ds, hrus, rivers, bg, lv95)

# Prepare arguments
args_list = [(i, t, ds, hrus, rivers, bg, lv95) for i, t in enumerate(ds.time.sel(time='2023'))]

with ThreadPoolExecutor(max_workers=16) as executor:
    list(tqdm(executor.map(plot_wrapper, args_list), total=len(args_list), ncols=120))

# Combine into video
a = '_zh' if ZH else '_ch'
p = "./figures/anim/*_zh.png" if ZH else './figures/anim/*_ch.png'
os.system(
    f"/usr/bin/ffmpeg -y -framerate 10 -pattern_type glob -i '{p}' "
    f"-c:v libx264 -vf \"pad=ceil(iw/2)*2:ceil(ih/2)*2\" -pix_fmt yuv420p './figures/anim/animation{a}.mp4'"
)


In [ ]:
q_cmap = 'inferno'
q_norm = mpl.colors.LogNorm(vmin=1, vmax=50)
q_alpha = 0.5

Q_cmap = 'YlGnBu_r'
Q_norm = mpl.colors.LogNorm(vmin=0.1, vmax=3000)
Q_alpha = 1.0

S_cmap = 'BrBG'
S_norm = mpl.colors.Normalize(vmin=0, vmax=1)
S_alpha = 0.5

times = ds.time.sel(time=['2023-01-01'])

for i, time in tqdm(enumerate(times), total=len(times), ncols=120):

    fig, ax = plt.subplots(figsize=(7, 7) if ZH else (7, 4), subplot_kw={'projection': lv95, 'frameon': True})

    ax.set_global()

    dss = ds.sel(time=time)
    hrus_Q = enrich(dss.Q_pred_m3s, hrus, 'Q')

    cmap = 'Greys'
    norm = mpl.colors.Normalize(vmin=0.0, vmax=50)
    alpha = 0.4

    hrus_p = enrich(dss.prec, hrus, 'p')

    hrus_p.plot(
        column='p',
        ec='w',
        lw=0.0,
        norm=norm,
        cmap=cmap,
        alpha=alpha,
        antialiased=True,
        legend=False,
        ax=ax,
        rasterized=True
    )

    axins = ax.inset_axes((0.03, 0.93, 0.18, 0.02), zorder=25)
    mappable = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    cbar = plt.colorbar(
        mappable,
        cax=axins,
        orientation='horizontal',   # or 'vertical'
        extend='max',
        label='Precipitation (mm/d)'
    )
    cbar.ax.xaxis.label.set_size(7)
    cbar.ax.tick_params(labelsize=7)
    cbar_halo(cbar=cbar, linewidth=0, labelsize=6)

    # lakes = hrus_q.loc[ds.hru.where(ds.is_lake.load() & (ds.area.load() > 2), drop=True)]

    # lakes.plot(
    #     ec='k',
    #     fc='w',
    #     lw=0.2,
    #     alpha=1,
    #     legend=False,
    #     zorder=30,
    #     ax=ax,
    #     rasterized=True
    # )
    # lakes.plot(
    #     column='classes',
    #     ec='w',
    #     lw=0.0,
    #     cmap=cmap,
    #     alpha=alpha,
    #     legend=False,
    #     zorder=30,
    #     ax=ax,
    #     rasterized=True
    # )

    # rivers_cut = gpd.overlay(
    #     rivers,
    #     hrus_Q.reset_index()[['EZGNR', 'Q', 'geometry']],
    #     how='intersection'
    # )
    # rivers_Q = rivers_cut.explode(index_parts=False).reset_index(drop=True)
    # rivers_Q = rivers_Q.sort_values('Q', ascending=True)
    # rw = np.clip(np.log(rivers_Q.CUM_LEN + 1) / 15 - 0.1, 0.3, None)

    bg.plot.imshow(ax=ax, interpolation='bilinear', alpha=0.5, zorder=0)

    hrus_T = enrich(dss, hrus, 'Q_ano')

    rivers_cut = gpd.overlay(
        rivers,
        hrus_T.reset_index()[['EZGNR', 'Q_ano', 'geometry']],
        how='intersection'
    )
    rivers_Q = rivers_cut.explode(index_parts=False).reset_index(drop=True)

    rivers_Q = rivers_Q.sort_values('Q_ano', ascending=True)

    rw = np.clip(np.log(rivers_Q.CUM_LEN + 1) / 15 - 0.1, 0.3, None)

    # cmap = sns.diverging_palette(240, 20, as_cmap=True)
    # cmap = cmap_new
    cmap = 'coolwarm'

    # norm = mpl.colors.LogNorm(vmin=0.1, vmax=3000)
    norm = mpl.colors.Normalize(vmin=-100, vmax=100)

    rivers_Q.plot(
        color='0.3',
        linewidth=rw + 0.2,
        zorder=2,
        capstyle='round',
        legend=False,
        ax=ax)

    rivers_Q.plot(
        column='Q_ano',
        linewidth=rw - 0.3,
        norm=norm,
        zorder=2,
        capstyle='round',
        fc='none',
        alpha=1.0,
        cmap=cmap,
        legend=False,
        ax=ax)

    axins = ax.inset_axes((0.03, 0.80, 0.18, 0.02), zorder=25)
    mappable = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    cbar = plt.colorbar(
        mappable,
        cax=axins,
        orientation='horizontal',   # or 'vertical'
        extend='neither',
        label='Relative discharge'
    )

    cbar.set_ticks([-100, 0, 100])
    cbar.ax.set_xticklabels(['very\nlow', 'normal', 'very\nhigh'])

    cbar_halo(cbar=cbar, linewidth=0, labelsize=6)

    date = dss.time.dt.strftime('%Y-%m-%d').item()
    ax.text(0.99, 0.01, f'{date}', fontsize=8, transform=ax.transAxes,
        verticalalignment='bottom', horizontalalignment='right',
        zorder=40)

    ax.set_title('')

    fig.savefig(f'./figures/anim/{i:04d}.png', dpi=300, bbox_inches='tight')
 
    plt.close()

a = '_zh' if ZH else ''
s = (
    "ffmpeg -y -framerate 8 -pattern_type glob -i './figures/anim/*.png' -c:v libx264 -pix_fmt yuv420p "
    f"'./figures/anim/animation{a}.mp4'"
)
os.system(s)

In [ ]:
hrus[hru_mask].plot()

In [ ]:
hrus.columns